# **Day 36: JSON and GeoJSON**

## **What is JSON?**
**JSON** (JavaScript Object Notation) is a text format for storing and sharing data. Almost every website and app sends data in JSON. It looks almost the same as a Python dictionary.

| Python | JSON |
|---|---|
| dict | object `{ }` |
| list, tuple | array `[ ]` |
| str | string |
| int, float | number |
| True / False | true / false |
| None | null |

The `json` module has four main functions:
- `json.dumps()`: Python **to** JSON text (**s** = string)
- `json.loads()`: JSON text **to** Python
- `json.dump()`: Python to a JSON **file**
- `json.load()`: JSON **file** to Python

In [1]:
import json

trip = {
    "name": "North Bengal trip",
    "days": 5,
    "places": ["Siliguri", "Darjeeling", "Kalimpong"],
    "budget": 18500.50,
    "booked": True,
    "guide": None,
}

text = json.dumps(trip)
print(text)
print(type(text))

{"name": "North Bengal trip", "days": 5, "places": ["Siliguri", "Darjeeling", "Kalimpong"], "budget": 18500.5, "booked": true, "guide": null}
<class 'str'>


In [2]:
# Pretty printing with indent
print(json.dumps(trip, indent=2))

{
  "name": "North Bengal trip",
  "days": 5,
  "places": [
    "Siliguri",
    "Darjeeling",
    "Kalimpong"
  ],
  "budget": 18500.5,
  "booked": true,
  "guide": null
}


In [3]:
# JSON text back to Python
back = json.loads(text)
print(back["places"][1], type(back))

Darjeeling <class 'dict'>


## **Saving and Loading a JSON File**

In [4]:
with open("trip.json", "w", encoding="utf-8") as f:
    json.dump(trip, f, indent=2)

with open("trip.json", encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded == trip)

True


## **Bengali Text in JSON**
By default, non-English letters are saved as codes like `ক`. Use `ensure_ascii=False` to keep them readable.

In [5]:
d = {"city": "কলকাতা", "river": "হুগলি"}
print(json.dumps(d))
print(json.dumps(d, ensure_ascii=False))

{"city": "\u0995\u09b2\u0995\u09be\u09a4\u09be", "river": "\u09b9\u09c1\u0997\u09b2\u09bf"}
{"city": "কলকাতা", "river": "হুগলি"}


## **What is GeoJSON?**
**GeoJSON** is JSON made for **maps**. It is one of the most common formats in GIS. QGIS, Google Maps, Leaflet and GeoPandas can all read it.

A GeoJSON file is a **FeatureCollection** (a group of features). Each **Feature** has:
1. **geometry:** the shape and location. The main types are:
   - **Point** (one location, like a school)
   - **LineString** (a line, like a road or a river)
   - **Polygon** (an area, like a district or a lake)
2. **properties:** the information about it (name, population and so on).

**Remember:** Coordinates in GeoJSON are always **[longitude, latitude]**.

## **Creating a GeoJSON File from Python**

In [6]:
cities = [
    ("Kolkata", 22.5726, 88.3639, 4496694),
    ("Siliguri", 26.7271, 88.3953, 513264),
    ("Durgapur", 23.5204, 87.3119, 566517),
    ("Midnapore", 22.4257, 87.3199, 169127),
]

features = []
for name, lat, lon, pop in cities:
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lon, lat]},
        "properties": {"name": name, "population": pop},
    })

# A line: a simple route from Kolkata to Digha
features.append({
    "type": "Feature",
    "geometry": {"type": "LineString",
                 "coordinates": [[88.36, 22.57], [88.10, 22.35], [87.92, 22.30], [87.51, 21.63]]},
    "properties": {"name": "Kolkata - Digha route"},
})

geojson = {"type": "FeatureCollection", "features": features}

with open("wb_places.geojson", "w", encoding="utf-8") as f:
    json.dump(geojson, f, indent=2)

print("Saved", len(features), "features")
print(json.dumps(features[0], indent=2))

Saved 5 features
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      88.3639,
      22.5726
    ]
  },
  "properties": {
    "name": "Kolkata",
    "population": 4496694
  }
}


This `.geojson` file can be dragged directly into **QGIS** or opened on the website **geojson.io** to see the points and the line on a real map.

## **Reading a GeoJSON File**

In [7]:
with open("wb_places.geojson", encoding="utf-8") as f:
    data = json.load(f)

for feat in data["features"]:
    geom_type = feat["geometry"]["type"]
    name = feat["properties"]["name"]
    if geom_type == "Point":
        lon, lat = feat["geometry"]["coordinates"]
        print(f"Point      : {name} at ({lat}, {lon})")
    else:
        n = len(feat["geometry"]["coordinates"])
        print(f"{geom_type:<11}: {name} with {n} points")

Point      : Kolkata at (22.5726, 88.3639)
Point      : Siliguri at (26.7271, 88.3953)
Point      : Durgapur at (23.5204, 87.3119)
Point      : Midnapore at (22.4257, 87.3199)
LineString : Kolkata - Digha route with 4 points


Let us understand the code above:
- `data["features"]` is the list of all features.
- We check the geometry type, because a Point has one coordinate pair but a LineString has many.

## **Things JSON Cannot Store Directly**

In [8]:
from datetime import date

# json.dumps({"date": date(2024, 10, 10)})     # This will raise a TypeError

# Solution: convert the date to text first
print(json.dumps({"date": date(2024, 10, 10).isoformat()}))

{"date": "2024-10-10"}


## **Practice Problem: Weather API Response**
This is a made-up response from a weather website. Let us read it.

In [9]:
response = '''
{
  "city": "Kolkata",
  "forecast": [
    {"day": "Mon", "max": 34, "min": 27, "rain_chance": 60},
    {"day": "Tue", "max": 33, "min": 26, "rain_chance": 80},
    {"day": "Wed", "max": 35, "min": 27, "rain_chance": 20}
  ]
}
'''
data = json.loads(response)
for f in data["forecast"]:
    umbrella = "Take an umbrella" if f["rain_chance"] >= 60 else ""
    print(f"{f['day']}: {f['min']}-{f['max']} C, rain {f['rain_chance']}%  {umbrella}")

Mon: 27-34 C, rain 60%  Take an umbrella
Tue: 26-33 C, rain 80%  Take an umbrella
Wed: 27-35 C, rain 20%  


In [10]:
# Quick question
# Guess the output of this code cell
# import json
# print(json.dumps({"a": True, "b": None, "c": (1, 2)}))